# MyceliumFractalNet — Interactive Analysis

**Detect critical transitions. Explain why. Intervene before collapse.**

This notebook demonstrates MFN's core value: a spatial dynamical system is simulated,
diagnosed across 10+ metrics simultaneously, and if it's approaching a tipping point —
MFN proposes the minimal intervention to prevent it, with causal proof at every step.

---

In [ ]:
import numpy as np
import mycelium_fractal_net as mfn
from mycelium_fractal_net.core.unified_engine import UnifiedEngine

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import plotly.io as pio
pio.templates.default = 'plotly_dark'

print(f'MFN v{mfn.__version__}')

## 1. Simulate Two Systems

Same initial conditions, different parameters.
**Stable**: default diffusion. **Stressed**: high diffusion + noise → approaching collapse.

In [ ]:
# Stable system
stable = mfn.simulate(mfn.SimulationSpec(grid_size=32, steps=60, seed=42))

# Stressed system: high diffusion + noise
stressed = mfn.simulate(mfn.SimulationSpec(
    grid_size=32, steps=60, seed=42,
    alpha=0.24, jitter_var=0.008, quantum_jitter=True,
))

print(f'Stable field:   [{stable.field.min():.4f}, {stable.field.max():.4f}]')
print(f'Stressed field: [{stressed.field.min():.4f}, {stressed.field.max():.4f}]')

### Field Morphology Comparison

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Stable — Turing Pattern', 'Stressed — Pattern Collapse'],
    horizontal_spacing=0.08,
)

common = dict(colorscale='Viridis', zmin=-0.095, zmax=0.04, colorbar=dict(title='V'))
fig.add_trace(go.Heatmap(z=stable.field, **common, showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=stressed.field, **common), row=1, col=2)

fig.update_layout(
    height=400, width=850,
    title_text='Membrane Potential Field (mV)',
    margin=dict(t=60, b=20),
)
fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.show()

### Temporal Evolution

In [ ]:
# Mean field over time
mean_stable = stable.history.mean(axis=(1, 2))
mean_stressed = stressed.history.mean(axis=(1, 2))
std_stable = stable.history.std(axis=(1, 2))
std_stressed = stressed.history.std(axis=(1, 2))
t = np.arange(len(mean_stable))

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Mean Potential', 'Spatial Variance (pattern strength)'],
)

fig.add_trace(go.Scatter(x=t, y=mean_stable*1000, name='Stable', line=dict(color='#00CC96', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=mean_stressed*1000, name='Stressed', line=dict(color='#EF553B', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=t, y=std_stable*1000, name='Stable', line=dict(color='#00CC96', width=2), showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=t, y=std_stressed*1000, name='Stressed', line=dict(color='#EF553B', width=2), showlegend=False), row=1, col=2)

fig.update_xaxes(title_text='Step', row=1, col=1)
fig.update_xaxes(title_text='Step', row=1, col=2)
fig.update_yaxes(title_text='mV', row=1, col=1)
fig.update_yaxes(title_text='mV', row=1, col=2)
fig.update_layout(height=350, width=850, margin=dict(t=40, b=40))
fig.show()

---
## 2. Full Diagnosis

One call → anomaly detection, early warning, causal validation, basin stability,
fractal analysis, persuadability assessment.

In [ ]:
engine = UnifiedEngine()

report_stable = engine.analyze(stable)
report_stressed = engine.analyze(stressed)

print('STABLE:')
print(report_stable.summary())
print(report_stable.interpretation())
print()
print('STRESSED:')
print(report_stressed.summary())
print(report_stressed.interpretation())

### Diagnostic Radar

In [ ]:
categories = [
    'Anomaly\nScore', 'EWS', 'Basin\nStability',
    'Multifractal\nΔα (norm)', 'Hurst\n(norm)', 'χ Invariant',
]

def normalize(val, vmin, vmax):
    return max(0, min(1, (val - vmin) / (vmax - vmin + 1e-12)))

vals_stable = [
    report_stable.anomaly_score,
    report_stable.ews_score,
    report_stable.basin_stability,
    normalize(report_stable.delta_alpha, 0, 5),
    normalize(report_stable.hurst_exponent, 0, 3),
    report_stable.chi_invariant,
]
vals_stressed = [
    report_stressed.anomaly_score,
    report_stressed.ews_score,
    report_stressed.basin_stability,
    normalize(report_stressed.delta_alpha, 0, 5),
    normalize(report_stressed.hurst_exponent, 0, 3),
    report_stressed.chi_invariant,
]

fig = go.Figure()
fig.add_trace(go.Scatterpolar(
    r=vals_stable + [vals_stable[0]], theta=categories + [categories[0]],
    fill='toself', name='Stable',
    fillcolor='rgba(0, 204, 150, 0.15)', line=dict(color='#00CC96', width=2),
))
fig.add_trace(go.Scatterpolar(
    r=vals_stressed + [vals_stressed[0]], theta=categories + [categories[0]],
    fill='toself', name='Stressed',
    fillcolor='rgba(239, 85, 59, 0.15)', line=dict(color='#EF553B', width=2),
))
fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    title='Diagnostic Fingerprint',
    height=450, width=550,
    margin=dict(t=60, b=20),
)
fig.show()

### Metric Comparison Bar Chart

In [ ]:
metrics = {
    'Anomaly Score': (report_stable.anomaly_score, report_stressed.anomaly_score),
    'EWS Score': (report_stable.ews_score, report_stressed.ews_score),
    'Basin Stability': (report_stable.basin_stability, report_stressed.basin_stability),
    'Δα (multifractal)': (report_stable.delta_alpha, report_stressed.delta_alpha),
    'Hurst Exponent': (report_stable.hurst_exponent, report_stressed.hurst_exponent),
    'χ Invariant': (report_stable.chi_invariant, report_stressed.chi_invariant),
    'Lacunarity(4)': (report_stable.lacunarity_4, report_stressed.lacunarity_4),
}

names = list(metrics.keys())
v_stable = [metrics[n][0] for n in names]
v_stressed = [metrics[n][1] for n in names]

fig = go.Figure()
fig.add_trace(go.Bar(name='Stable', x=names, y=v_stable, marker_color='#00CC96'))
fig.add_trace(go.Bar(name='Stressed', x=names, y=v_stressed, marker_color='#EF553B'))
fig.update_layout(
    barmode='group', height=400, width=850,
    title='Metric Comparison: Stable vs Stressed',
    yaxis_title='Value',
    margin=dict(t=50, b=80),
)
fig.show()

---
## 3. Pattern Evolution Film Strip

Watch the field evolve frame-by-frame. Stable system develops Turing patterns;
stressed system loses spatial structure.

In [ ]:
frames_to_show = [0, 10, 20, 30, 40, 59]

fig = make_subplots(
    rows=2, cols=len(frames_to_show),
    subplot_titles=[f't={t}' for t in frames_to_show] * 2,
    vertical_spacing=0.08, horizontal_spacing=0.02,
)

for i, t in enumerate(frames_to_show):
    fig.add_trace(go.Heatmap(
        z=stable.history[t], colorscale='Viridis',
        zmin=-0.095, zmax=0.04, showscale=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Heatmap(
        z=stressed.history[t], colorscale='Viridis',
        zmin=-0.095, zmax=0.04, showscale=(i == len(frames_to_show)-1),
    ), row=2, col=i+1)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)

fig.add_annotation(text='Stable', x=-0.02, y=0.75, xref='paper', yref='paper',
                   textangle=-90, showarrow=False, font=dict(size=14, color='#00CC96'))
fig.add_annotation(text='Stressed', x=-0.02, y=0.25, xref='paper', yref='paper',
                   textangle=-90, showarrow=False, font=dict(size=14, color='#EF553B'))

fig.update_layout(height=350, width=900, title_text='Pattern Evolution', margin=dict(t=50, l=40, b=10))
fig.show()

---
## 4. Early Warning Timeline

Monitor the stressed system tick-by-tick. MFN detects the anomaly
at every checkpoint — before visible collapse.

In [ ]:
spec_stressed = mfn.SimulationSpec(
    grid_size=32, steps=60, seed=42,
    alpha=0.24, jitter_var=0.008, quantum_jitter=True,
)
ticks = mfn.watch(spec_stressed, n_steps_per_tick=10, n_ticks=6)

ews_scores = [t.warning.ews_score for t in ticks]
anomaly_scores = [t.anomaly.score for t in ticks]
severities = [t.severity for t in ticks]
tick_labels = [f'{i*10}-{(i+1)*10}' for i in range(len(ticks))]

colors = ['#EF553B' if s in ('warning', 'critical') else '#636EFA' for s in severities]

fig = make_subplots(rows=1, cols=2, subplot_titles=['Anomaly Score', 'EWS Score'])

fig.add_trace(go.Bar(
    x=tick_labels, y=anomaly_scores, marker_color=colors, name='Anomaly',
    text=[f'{s:.2f}' for s in anomaly_scores], textposition='outside',
), row=1, col=1)

fig.add_trace(go.Bar(
    x=tick_labels, y=ews_scores, marker_color='#AB63FA', name='EWS',
    text=[f'{s:.3f}' for s in ews_scores], textposition='outside',
), row=1, col=2)

fig.update_xaxes(title_text='Steps')
fig.update_yaxes(range=[0, 0.7], row=1, col=1)
fig.update_yaxes(range=[0, 0.1], row=1, col=2)
fig.update_layout(
    height=350, width=850,
    title_text='Early Warning Timeline (Stressed System)',
    showlegend=False,
    margin=dict(t=60, b=40),
)
fig.show()

for i, t in enumerate(ticks):
    marker = '\u26a0' if t.severity in ('warning', 'critical') else '\u2713'
    print(f'  step {i*10:2d}-{(i+1)*10:2d}: {marker} {t.severity:8s} anomaly={t.anomaly.label}({t.anomaly.score:.2f})')

---
## 5. Intervention Planning

MFN finds the minimal parameter change to restore the stressed system
to nominal regime — with causal validation at every step.

In [ ]:
plan = mfn.plan_intervention(stressed)
best = plan.best_candidate

if best is not None:
    print(f'Viable plan: {plan.has_viable_plan}')
    print(f'Candidates evaluated: {len(plan.candidates)}')
    print(f'Pareto front size: {len(plan.pareto_front)}')
    print(f'Composite score: {best.composite_score:.3f}')
    print(f'Causal gate: {best.causal_decision}')
    if best.detection_after:
        print(f'Anomaly after intervention: {best.detection_after.label} ({best.detection_after.score:.3f})')
    print()
    print('Proposed changes:')
    for spec in best.proposed_changes:
        delta = spec.proposed_value - spec.current_value
        if abs(delta) > 1e-6:
            direction = '\u2191' if delta > 0 else '\u2193'
            print(f'  {spec.name:30s} {spec.current_value:8.4f} {direction} {spec.proposed_value:8.4f}  (\u0394={delta:+.4f})')
else:
    print('No viable intervention found.')

In [ ]:
# Visualize intervention as waterfall
if best is not None:
    changed = [(s.name, s.current_value, s.proposed_value)
               for s in best.proposed_changes
               if abs(s.proposed_value - s.current_value) > 1e-6]

    if changed:
        names_ch = [c[0] for c in changed]
        current = [c[1] for c in changed]
        proposed = [c[2] for c in changed]

        fig = go.Figure()
        fig.add_trace(go.Bar(name='Before', x=names_ch, y=current, marker_color='#EF553B'))
        fig.add_trace(go.Bar(name='After', x=names_ch, y=proposed, marker_color='#00CC96'))
        fig.update_layout(
            barmode='group', height=400, width=700,
            title='Intervention: Parameter Changes',
            yaxis_title='Value',
            margin=dict(t=50, b=80),
        )
        fig.show()

---
## 6. Bio Layer — Transport Network

Physarum conductivity network adapts to the field.
Where transport is efficient, hyphae grow faster.

In [ ]:
from mycelium_fractal_net.bio import BioExtension

bio = BioExtension.from_sequence(stable).step(n=5)
report_bio = bio.report()
print(report_bio.summary())

# Conductivity map
D_h = bio.physarum_state.D_h
D_v = bio.physarum_state.D_v

# Convert to per-cell conductivity (mean of adjacent edges)
N = stable.field.shape[0]
cond = np.zeros((N, N))
count = np.zeros((N, N))
cond[:, :-1] += D_h; cond[:, 1:] += D_h; count[:, :-1] += 1; count[:, 1:] += 1
cond[:-1, :] += D_v; cond[1:, :] += D_v; count[:-1, :] += 1; count[1:, :] += 1
cond_cell = cond / np.maximum(count, 1)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Field (mV)', 'Physarum Conductivity', 'Hyphal Density'],
    horizontal_spacing=0.06,
)
fig.add_trace(go.Heatmap(z=stable.field, colorscale='Viridis', showscale=False), row=1, col=1)
fig.add_trace(go.Heatmap(z=cond_cell, colorscale='Inferno', showscale=False), row=1, col=2)
fig.add_trace(go.Heatmap(z=bio.anastomosis_state.B, colorscale='Cividis', showscale=False), row=1, col=3)

fig.update_xaxes(showticklabels=False)
fig.update_yaxes(showticklabels=False)
fig.update_layout(height=300, width=850, margin=dict(t=40, b=10))
fig.show()

---
## 7. Causal Validation

Every MFN conclusion passes through 46 executable causal rules.
No result escapes without proof.

In [ ]:
diag = mfn.diagnose(stressed)
causal = diag.causal
td = causal.to_dict()

print(f'Decision:      {causal.decision.value}')
print(f'Rules checked: {td["total_rules"]}')
print(f'Passed:        {td["passed_rules"]}')
print(f'Warnings:      {causal.warning_count}')
print(f'Errors:        {causal.error_count}')
print(f'Pass rate:     {td["passed_rules"] / max(td["total_rules"], 1):.1%}')

# Show any violations
if causal.violations:
    print('\nViolations:')
    for v in causal.violations[:5]:
        print(f'  [{v.severity}] {v.rule_id}: {v.message}')

---

## Summary

MFN provides **one unified pipeline** for spatial dynamical systems:

| Step | What | How |
|------|------|-----|
| **Simulate** | Generate morphogenetic field | Turing RD + neuromodulation |
| **Diagnose** | 10+ metrics simultaneously | TDA, multifractal, W2, basin stability, EWS |
| **Validate** | Causal proof for every conclusion | 46 executable rules |
| **Intervene** | Minimal change to shift regime | Counterfactual search + Pareto front |

No other tool does all four in one call.